In [1]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [2]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [2]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [3]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course':'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )
    return results


In [5]:
question = 'Can I still join the course?'

In [6]:
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(query, search_results):
    context=""

    for doc in search_results:
        context= context + f"section: {doc['section']}\nquestion:{doc['question']}\nanswer: {doc['text']}\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [10]:
search_results = search(question)

In [12]:
search_results[0]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp',
 '_id': 2}

In [12]:
prompt = build_prompt(question, search_results)

In [4]:
from openai import OpenAI
import os
api_key = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key,
)

In [ ]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [15]:
answer = llm(prompt)

In [16]:
answer

"Yes, you can still join the course after the start date. Even if you don't register, you're still eligible to submit the homeworks. Just be aware that there will be deadlines for turning in the final projects, so make sure not to leave everything for the last minute."

In [17]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [18]:
rag("How do I patch KDE under FreeBSD?")

'The provided context does not contain any information on how to patch KDE under FreeBSD. Please refer to appropriate FreeBSD or KDE documentation for assistance with this topic.'

AGENTIC RAG

In [19]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
{question}
</QUESTION>

<CONTEXT> 
{context}
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database.
In this case, use the following output template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}
""".strip()

In [20]:
question = 'Can I still join the course?'
context = 'EMPTY'

In [21]:
prompt = prompt_template.format(question=question, context=context)

In [22]:
answer_json = llm(prompt)

In [23]:
answer_json

'{\n"action": "SEARCH",\n"reasoning": "The question is about whether the student can still join the course, and there is currently no context provided to determine the enrollment status or deadlines."\n}'

In [24]:
import json
answer = json.loads(answer_json)

In [25]:
answer

{'action': 'SEARCH',
 'reasoning': 'The question is about whether the student can still join the course, and there is currently no context provided to determine the enrollment status or deadlines.'}

In [8]:
def build_context(search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    return context.strip()

In [27]:
search_results = search(question)
context = build_context(search_results)
prompt = prompt_template.format(question=question, context=context)

In [28]:
answer_json = llm(prompt)

In [30]:
print(answer_json)

{
"action": "ANSWER",
"answer": "Yes, you can still join the course even after the start date. You are eligible to submit homework assignments, but keep in mind that there will be deadlines for the final projects, so it's advisable not to leave everything until the last minute.",
"source": "CONTEXT"
}


AGENTIC SEARCH

In [9]:
def dedup(seq):
    seen = set()
    result = []
    for el in seq:
        _id = el['_id']
        if _id in seen:
            continue
        seen.add(_id)
        result.append(el)
    return result

In [10]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than {max_iterations} iterations for a given student question.
The current iteration number: {iteration_number}. If we exceed the allowed number 
of iterations, give the best possible answer with the provided information.

Output templates:

If you want to perform search, use this template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>",
"keywords": ["search query 1", "search query 2", ...]
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER_CONTEXT",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}

<QUESTION>
{question}
</QUESTION>

<SEARCH_QUERIES>
{search_queries}
</SEARCH_QUERIES>

<CONTEXT> 
{context}
</CONTEXT>

<PREVIOUS_ACTIONS>
{previous_actions}
</PREVIOUS_ACTIONS>
""".strip()

In [11]:
question = 'how do I do well on module 1'
max_iterations = 3
iteration_number =0
search_queries = []
search_results =[] 
previous_actions = [] 

In [12]:
context = build_context(search_results)

prompt = prompt_template.format(
    question = question,
    context = context,
    search_queries="\n".join(search_queries),
    previous_actions = "\n".join([json.dumps(a) for a in previous_actions]),
    max_iterations = max_iterations,
    iteration_number = iteration_number
)    

In [13]:
answer_json = llm(prompt)

In [14]:
answer_json

'{\n"action": "SEARCH",\n"reasoning": "To provide the student with tailored advice on how to excel in Module 1, I need to gather specific strategies and tips that may have been covered in the FAQs. This will assist in offering guidance that is relevant and constructive.",\n"keywords": ["how to succeed in module 1", "tips for Module 1", "study strategies for Module 1"]\n}'

In [15]:
import json
answer = json.loads(answer_json)

In [16]:
answer

{'action': 'SEARCH',
 'reasoning': 'To provide the student with tailored advice on how to excel in Module 1, I need to gather specific strategies and tips that may have been covered in the FAQs. This will assist in offering guidance that is relevant and constructive.',
 'keywords': ['how to succeed in module 1',
  'tips for Module 1',
  'study strategies for Module 1']}

In [17]:
previous_actions.append(answer)

In [18]:
previous_actions

[{'action': 'SEARCH',
  'reasoning': 'To provide the student with tailored advice on how to excel in Module 1, I need to gather specific strategies and tips that may have been covered in the FAQs. This will assist in offering guidance that is relevant and constructive.',
  'keywords': ['how to succeed in module 1',
   'tips for Module 1',
   'study strategies for Module 1']}]

In [19]:
keywords = answer['keywords']
search_queries

[]

In [20]:
for kw in keywords:
    search_queries.append(kw)
    sr = search(kw)
    search_results.extend(sr)

In [21]:
search_results[0]

{'text': 'Even after installing pyspark correctly on linux machine (VM ) as per course instructions, faced a module not found error in jupyter notebook .\nThe solution which worked for me(use following in jupyter notebook) :\n!pip install findspark\nimport findspark\nfindspark.init()\nThereafter , import pyspark and create spark contex<<t as usual\nNone of the solutions above worked for me till I ran !pip3 install pyspark instead !pip install pyspark.\nFilter based on conditions based on multiple columns\nfrom pyspark.sql.functions import col\nnew_final.filter((new_final.a_zone=="Murray Hill") & (new_final.b_zone=="Midwood")).show()\nKrishna Anand',
 'section': 'Module 5: pyspark',
 'question': 'Module Not Found Error in Jupyter Notebook .',
 'course': 'data-engineering-zoomcamp',
 '_id': 322}

In [22]:
search_results = dedup(search_results)

In [23]:
iteration_number = 2

context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=max_iterations,
    iteration_number=iteration_number
)

In [65]:
prompt

'You\'re a course teaching assistant.\n\nYou\'re given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.\n\nThe CONTEXT is build with the documents from our FAQ database.\nSEARCH_QUERIES contains the queries that were used to retrieve the documents\nfrom FAQ to and add them to the context.\nPREVIOUS_ACTIONS contains the actions you already performed.\n\nAt the beginning the CONTEXT is empty.\n\nYou can perform the following actions:\n\n- Search in the FAQ database to get more data for the CONTEXT\n- Answer the question using the CONTEXT\n- Answer the question using your own knowledge\n\nFor the SEARCH action, build search requests based on the CONTEXT and the QUESTION.\nCarefully analyze the CONTEXT and generate the requests to deeply explore the topic. \n\nDon\'t use search queries used at the previous iterations.\n\nDon\'t repeat previously performed actions.\n\nDon\'t perform more than 3 iterations for a given student question

In [24]:
answer_json = llm(prompt)

In [25]:
answer = json.loads(answer_json)

In [26]:
print(answer)

{'action': 'SEARCH', 'reasoning': 'To further enhance my understanding and provide the best possible advice for succeeding in Module 1, I should look for specific study strategies, tips, or experiences shared by other students in the FAQs. This will enable me to compile comprehensive information on how to perform well in that module.', 'keywords': ['Module 1 study tips', 'succeed in Module 1', 'Module 1 best practices']}


In [ ]:
question = "what do I need to do to be successful at module 1?"

search_queries = []
search_results = []
previous_actions = []

iteration = 0

while True:
    print(f'ITERATION #{iteration}...')

    context = build_context(search_results)
    prompt = prompt_template.format(
        question=question,
        context=context,
        search_queries="\n".join(search_queries),
        previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
        max_iterations=3,
        iteration_number=iteration
    )

    print(prompt)

    answer_json = llm(prompt)
    answer = json.loads(answer_json)
    print(json.dumps(answer, indent=2))

    previous_actions.append(answer)

    action = answer['action']
    if action != 'SEARCH':
        break

    keywords = answer['keywords']
    search_queries = list(set(search_queries) | set(keywords))
    
    for k in keywords:
        res = search(k)
        search_results.extend(res)

    search_results = dedup(search_results)
    
    iteration = iteration + 1
    if iteration >= 4:
        break

    print()

FUNCTION CALLING(TOOL USE)

In [6]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [21]:
def do_call(tool_call_response):
    function_name = tool_call_response.name
    arguments= json.loads(tool_call_response.arguments)

    f = globals()[function_name]
    result = f(**arguments)

    return {
        "type": "function_call_output",
        "call_id": tool_call_response.call_id,
        "output": json.dumps(result, indent=2),
    }

In [11]:
question = "How do I do well in module 1?"

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
If you look up something in FAQ, convert the student question into multiple queries.
""".strip()

tools = [search_tool]

chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools =tools,
)
response.output

[ResponseFunctionToolCall(arguments='{"query": "how to succeed in module 1"}', call_id='call_prPjarDr3IZ0fqwCw8Hup0w7', name='search', type='function_call', id='fc_tmp_3wfn8f6d8so', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query": "tips for doing well in module 1"}', call_id='call_5yxiVl4mjjOq4zjyfjklEzOf', name='search', type='function_call', id='fc_tmp_li8318gjo6o', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query": "module 1 success strategies"}', call_id='call_QcVOnWxKzmqlnFaXw4EGB3GM', name='search', type='function_call', id='fc_tmp_n9wtkw9oapm', namespace=None, status='completed')]

In [12]:
calls = response.output

In [25]:
calls[0].type


'function_call'

In [22]:
for call in calls:
    result = do_call(call)
    chat_messages.append(call)
    chat_messages.append(result)

In [23]:
chat_messages

[{'role': 'developer',
  'content': "You're a course teaching assistant. \nYou're given a question from a course student and your task is to answer it.\nIf you look up something in FAQ, convert the student question into multiple queries."},
 {'role': 'user', 'content': 'How do I do well in module 1?'},
 ResponseFunctionToolCall(arguments='{"query": "how to succeed in module 1"}', call_id='call_prPjarDr3IZ0fqwCw8Hup0w7', name='search', type='function_call', id='fc_tmp_3wfn8f6d8so', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_prPjarDr3IZ0fqwCw8Hup0w7',
  'output': '[\n  {\n    "text": "Even after installing pyspark correctly on linux machine (VM ) as per course instructions, faced a module not found error in jupyter notebook .\\nThe solution which worked for me(use following in jupyter notebook) :\\n!pip install findspark\\nimport findspark\\nfindspark.init()\\nThereafter , import pyspark and create spark contex<<t as usual\\nNone of the solu

In [26]:
for entry in response.output:
    chat_messages.append(entry)
    print(entry.type)

    if entry.type == 'function_call':      
        result = do_call(entry)
        chat_messages.append(result)
    elif entry.type == 'message':
        print(entry.text) 

function_call
function_call
function_call


In [27]:
developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

Use FAQ if your own knowledge is not sufficient to answer the question.
When using FAQ, perform deep topic exploration: make one request to FAQ,
and then based on the results, make more requests.

At the end of each response, ask the user a follow up question based on your answer.
""".strip()

chat_messages = [
    {"role": "developer", "content": developer_prompt},
]

In [28]:
while True: # main Q&A loop
    question = input() # How do I do my best for module 1?
    if question == 'stop':
        break

    message = {"role": "user", "content": question}
    chat_messages.append(message)

    while True: # request-response loop - query API till get a message
        response = client.responses.create(
            model='gpt-4o-mini',
            input=chat_messages,
            tools=tools
        )

        has_messages = False
        
        for entry in response.output:
            chat_messages.append(entry)
        
            if entry.type == 'function_call':      
                print('function_call:', entry)
                print()
                result = do_call(entry)
                chat_messages.append(result)
            elif entry.type == 'message':
                print(entry.content[0].text)
                print()
                has_messages = True

        if has_messages:
            break

 How do i do my best for module 1


function_call: ResponseFunctionToolCall(arguments='{"query":"module 1 best practices"}', call_id='call_qOwzJnrph9joDp1KAMZzn6wB', name='search', type='function_call', id='fc_tmp_3s5x0sc2bly', namespace=None, status='completed')

To do your best in Module 1, here are some best practices and tips to keep in mind:

1. **Utilize Docker Efficiently**: Since Module 1 is focused on Docker and Terraform, it's important to familiarize yourself with Docker's best practices. Ensure that you are storing all your code in your default Linux distribution to optimize file system performance. This is especially significant for Windows 10 Home or Windows 11 Home users, where Docker relies on the WSL2 backend.

2. **Resolving Dependency Issues**: You might encounter errors like `ModuleNotFoundError` for Python modules. For example, if you're using SQLAlchemy and receive this error related to `psycopg2`, make sure to install the necessary modules using pip or conda. The command would be:
   ```
   pip ins

 how to utilize docker efficiently


function_call: ResponseFunctionToolCall(arguments='{"query":"utilize docker efficiently best practices"}', call_id='call_AVJwKcHANUQlWVgKMYhZ9Mf6', name='search', type='function_call', id='fc_tmp_2fqxu362iyf', namespace=None, status='completed')

Here are some best practices for utilizing Docker efficiently:

1. **Store Code in Linux Distro**: If you're using Docker on Windows 10 Home or Windows 11 Home, make sure you store all your code in your default Linux distribution. This improves file system performance since Docker runs on the WSL2 backend by default.

2. **Leverage Docker Compose**: Use Docker Compose to manage multi-container applications effectively. This allows you to define and run complex applications using a single `docker-compose.yml` file. Ensure that you properly specify the networks and services to avoid connection issues.

3. **Proper Volume Management**: When creating containers with persistent storage (volumes), be aware of file permissions. If you encounter issue

 stop
